:meth:`SeqMut.suggest` returns the top mutations that move a sequence toward the test-class CPP profile, ranked by ``shift_score`` (``sum sign(mean_dif) * ΔX``).

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
split_kws = sf.get_split_kws()
cpp = aa.CPP(df_parts=df_parts, split_kws=split_kws, verbose=False)
df_feat = cpp.run(labels=labels, n_filter=25)

seqm = aa.SeqMut()
aa.display_df(seqm.suggest(df_seq=df_seq, df_feat=df_feat, n=10, region="tmd"), n_rows=10, show_shape=True)

DataFrame shape: (10, 8)


,entry,pos,from_aa,to_aa,mutation,region,delta_cpp,shift_score
1,Q8IUW5,74,G,A,G74A,tmd,3.415670,3.415670
2,P05556,744,G,A,G744A,tmd,3.415670,3.415670
3,Q14802,52,G,A,G52A,tmd,3.415660,3.415660
4,P53801,112,G,A,G112A,tmd,3.415660,3.415660
5,Q8IUW5,74,G,E,G74E,tmd,2.904170,2.904170
6,P05556,744,G,E,G744E,tmd,2.904170,2.904170
7,Q14802,52,G,E,G52E,tmd,2.904160,2.904160
8,P53801,112,G,E,G112E,tmd,2.904160,2.904160
9,Q8IUW5,78,C,A,C78A,tmd,2.859590,2.859590
10,P01135,118,C,A,C118A,tmd,2.859580,2.859580


### Further parameters: `to_aa`, `weight`, `jmd_n_len`, `jmd_c_len`

`to_aa` restricts the substitution alphabet, `weight` scales the shift score by a `df_feat` column (`'abs_auc'` or `'feat_importance'`) so more discriminative features count more, and `jmd_n_len` / `jmd_c_len` set the JMD lengths of the split geometry.

In [2]:
# Weighted suggestion over a restricted alphabet on the default 10/10 split geometry
df_suggest = seqm.suggest(df_seq=df_seq, df_feat=df_feat, n=10, region="tmd",
                            to_aa=["A", "L", "V", "P"], weight="abs_auc",
                            jmd_n_len=10, jmd_c_len=10)
aa.display_df(df_suggest, n_rows=10, show_shape=True)

DataFrame shape: (10, 8)


,entry,pos,from_aa,to_aa,mutation,region,delta_cpp,shift_score
1,Q8IUW5,74,G,A,G74A,tmd,3.415670,1.557450
2,P05556,744,G,A,G744A,tmd,3.415670,1.557450
3,P53801,112,G,A,G112A,tmd,3.415660,1.557446
4,Q14802,52,G,A,G52A,tmd,3.415660,1.557446
5,Q8IUW5,78,C,A,C78A,tmd,2.859590,1.300775
6,P01135,118,C,A,C118A,tmd,2.859580,1.300771
7,Q969W9,60,C,A,C60A,tmd,2.859580,1.300771
8,P53801,116,C,A,C116A,tmd,2.859580,1.300771
9,P53801,112,G,L,G112L,tmd,2.801250,1.278101
10,P05556,744,G,L,G744L,tmd,2.801250,1.278101


**Design constraints.** `constraints` takes a shared `DesignConstraints` object that replaces the `region` / `to_aa` shorthands and additionally excludes immutable positions and forbidden target residues from the ranking:

In [3]:
tmd_start = int(df_seq["tmd_start"].iloc[0])
dc = aa.DesignConstraints(mutable_positions="tmd",
                          permitted_substitutions=["A", "L", "V", "P"],
                          immutable_positions=[tmd_start],
                          forbidden_substitutions=["P"])
df_suggest_dc = seqm.suggest(df_seq=df_seq, df_feat=df_feat, n=10, constraints=dc)
aa.display_df(df_suggest_dc, n_rows=10, show_shape=True)


DataFrame shape: (10, 8)


,entry,pos,from_aa,to_aa,mutation,region,delta_cpp,shift_score
1,P05556,744,G,A,G744A,tmd,3.415670,3.415670
2,Q8IUW5,74,G,A,G74A,tmd,3.415670,3.415670
3,Q14802,52,G,A,G52A,tmd,3.415660,3.415660
4,P53801,112,G,A,G112A,tmd,3.415660,3.415660
5,Q8IUW5,78,C,A,C78A,tmd,2.859590,2.859590
6,P53801,116,C,A,C116A,tmd,2.859580,2.859580
7,P01135,118,C,A,C118A,tmd,2.859580,2.859580
8,Q969W9,60,C,A,C60A,tmd,2.859580,2.859580
9,P05556,744,G,L,G744L,tmd,2.801250,2.801250
10,Q14802,52,G,L,G52L,tmd,2.801250,2.801250
